In [8]:
from pathlib import Path

import pandas as pd

In [9]:
RESULT_BASES = [Path("results"), Path("code/backend/src/main/python/TEST/results")]

# True: 통합 크롤링(번개장터+중고나라). False: 번개장터 분리 CSV만
USE_INTEGRATED_CSV = True


def collect_glob(pattern):
    paths = []
    for base in RESULT_BASES:
        paths.extend(base.glob(pattern))
    return paths


csv_path = None
if USE_INTEGRATED_CSV:
    for base in RESULT_BASES:
        latest = base / "latest_crawling_no_filter.csv"
        if latest.exists():
            csv_path = latest
            break
    if csv_path is None:
        integrated = collect_glob("통합조회_전체_no_filter_*.csv")
        if integrated:
            csv_path = max(integrated, key=lambda path: path.stat().st_mtime)

if csv_path is None:
    bunjang_only = collect_glob("platform_번개장터_no_filter_*.csv")
    if bunjang_only:
        csv_path = max(bunjang_only, key=lambda path: path.stat().st_mtime)

if csv_path is None:
    raise FileNotFoundError(
        "CSV를 찾을 수 없습니다. results 폴더 또는 USE_INTEGRATED_CSV 설정을 확인하세요."
    )

df = pd.read_csv(csv_path, encoding="utf-8-sig")
print("로드 파일:", csv_path.resolve())
print(df["platform"].value_counts())
df.head()

로드 파일: C:\project\kdtproject\kdtproject\code\backend\src\main\python\TEST\results\latest_crawling_no_filter.csv
platform
번개장터    18526
중고나라     6781
Name: count, dtype: int64


,keyword,platform,pid,name,price,status,description,image_url,link,date,matched_keywords,canonical_name
0,게이밍 노트북,번개장터,407671617,최저가]ASUS 14인치 게이밍노트북 A14(8845HS/4060/램32,1270000,판매중,NaN,https://media.bunjang.co.kr/product/407671617_...,https://m.bunjang.co.kr/products/407671617,2026-05-12 20:54,게이밍 노트북,게이밍 노트북
1,게이밍 노트북,번개장터,396198529,레노버 LEGION 게이밍 노트북,329000,판매중,NaN,https://media.bunjang.co.kr/product/396198529_...,https://m.bunjang.co.kr/products/396198529,2026-05-12 20:50,게이밍 노트북,게이밍 노트북
2,게이밍 노트북,번개장터,399045782,"MSI i7 고성능 게이밍 노트북(램12GB,1.25TB,신품고가배터리)",529000,판매중,NaN,https://media.bunjang.co.kr/product/399045782_...,https://m.bunjang.co.kr/products/399045782,2026-05-12 20:42,게이밍 노트북,게이밍 노트북
3,게이밍 노트북,번개장터,407667568,[매입]미개봉 LG그램 갤럭시북 맥북 기업용 대량매입 게이밍 노트북,3000000,판매중,NaN,https://media.bunjang.co.kr/product/407667568_...,https://m.bunjang.co.kr/products/407667568,2026-05-12 20:34,"갤럭시, 게이밍 노트북",게이밍 노트북
4,게이밍 노트북,번개장터,407666216,[최저가]HP 16인치 게이밍노트북 오멘(라이젠9 365/5070/램32,2150000,판매중,NaN,https://media.bunjang.co.kr/product/407666216_...,https://m.bunjang.co.kr/products/407666216,2026-05-12 20:28,게이밍 노트북,게이밍 노트북


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25307 entries, 0 to 25306
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   keyword           25307 non-null  str    
 1   platform          25307 non-null  str    
 2   pid               25307 non-null  int64  
 3   name              25307 non-null  str    
 4   price             25307 non-null  int64  
 5   status            25307 non-null  str    
 6   description       0 non-null      float64
 7   image_url         25307 non-null  str    
 8   link              25307 non-null  str    
 9   date              25307 non-null  str    
 10  matched_keywords  25307 non-null  str    
 11  canonical_name    25307 non-null  str    
dtypes: float64(1), int64(2), str(9)
memory usage: 2.3 MB


## 계층 클러스터(L1→L2) · 노트북에서 테스트

[`cluster_by_platform_tokens.run_clustering`](cluster_by_platform_tokens.py) 결과를 **이 노트북만으로** 돌리고 표·요약·트리·텍스트 미리보기까지 확인합니다.

- 저장 위치: **`results/clusters/`** (`token_cluster_*` + `latest_*.csv`).
- **`CLUSTER_PREVIEW_ROWS`**: 숫자 → `df` 앞 N행만 입력(빠른 테스트). **`None`** → 크롤 `csv_path` 전체(시간 소요).
- **`CLUSTER_KWARGS`**: `threshold` / `threshold_l2` 등 실험 파라미터.
- 터미널과 동일한 요약·트리 텍스트는 아래 셀 맨 아래 **텍스트 미리보기**에서 출력합니다. (`CLUSTER_KWARGS`의 `print_stdout_preview=True`로 바꿔도 됩니다.)
- 작업 디렉터리가 프로젝트 루트여도 스크립트가 `TEST` 폴더의 모듈을 자동으로 찾습니다.
- CLI: **`python cluster_by_platform_tokens.py`** 저장 후 터미널 미리보기.


In [11]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

# Jupyter cwd가 프로젝트 루트여도 여기서 모듈을 찾도록 TEST 경로 추가
_here = Path.cwd().resolve()
for _test_root in (_here, _here / "code/backend/src/main/python/TEST"):
    _mod_path = (_test_root / "cluster_by_platform_tokens.py").resolve()
    if _mod_path.is_file():
        if str(_test_root.resolve()) not in sys.path:
            sys.path.insert(0, str(_test_root.resolve()))
        break
else:
    raise FileNotFoundError(
        "cluster_by_platform_tokens.py 를 찾지 못했습니다. "
        "노트북 cwd를 `.../TEST` 또는 프로젝트 루트로 두고 다시 실행하세요."
    )

from cluster_by_platform_tokens import DEFAULT_TOKEN_BLACKLIST_PATH, run_clustering, stdout_preview_clustering

# --- 테스트용 옵션 (여기만 바꿔 실험) ---
# CLUSTER_PREVIEW_ROWS: 숫면 df 앞쪽 N행만 입력 / None 이면 csv_path 통째로 (오래 걸릴 수 있음)
CLUSTER_PREVIEW_ROWS = 800

# run_clustering에 넘기는 인자 — 임계값 실험 시 수정
CLUSTER_KWARGS = dict(
    threshold=0.35,
    min_shared_tokens=1,
    threshold_l2=0.52,
    min_shared_tokens_l2=2,
    run_second_stage=True,
    token_blacklist_path=DEFAULT_TOKEN_BLACKLIST_PATH,
    print_stdout_preview=False,
)

cluster_out_dir = csv_path.resolve().parent / "clusters"
cluster_input = csv_path
preview_note = ""

if CLUSTER_PREVIEW_ROWS is None:
    preview_note = "(전체 행 입력: csv_path)"
else:
    preview_n = min(int(CLUSTER_PREVIEW_ROWS), len(df))
    cluster_input = csv_path.resolve().parent / "_cluster_preview_rows.csv"
    df.head(preview_n).to_csv(cluster_input, index=False, encoding="utf-8-sig")
    preview_note = f"(미리보기: df 상위 {preview_n}행)"

print("클러스터 입력:", cluster_input.resolve(), preview_note)
print("파라미터:", CLUSTER_KWARGS)

cluster_result = run_clustering(
    input_path=cluster_input,
    output_dir=cluster_out_dir,
    **CLUSTER_KWARGS,
)

summ_cols = ["platform", "cluster_l1_id", "cluster_id", "cluster_size", "breadcrumb_names", "derived_product_name", "sample_names"]
tree_cols = ["parent_node_id", "node_id", "platform", "depth", "item_count", "representative_name", "cluster_tokens"]
item_cols = ["platform", "pid", "name", "cluster_l1_id", "cluster_id", "breadcrumb_names", "derived_product_name"]

tree_df_pv = pd.read_csv(cluster_result["latest_tree_path"], encoding="utf-8-sig")
summ_df_pv = pd.read_csv(cluster_result["latest_summary_path"], encoding="utf-8-sig")
items_df_pv = pd.read_csv(cluster_result["latest_item_path"], encoding="utf-8-sig")

print(
    "\n저장:",
    "\n • items:", cluster_result["latest_item_path"],
    "\n • summary:", cluster_result["latest_summary_path"],
    "\n • tree:", cluster_result["latest_tree_path"],
)

display(Markdown("### 검증 요약"))
if summ_df_pv.empty:
    display(Markdown("요약(summary) 행이 없습니다."))
else:
    _vc = summ_df_pv.groupby("platform").size().rename("leaf_l2_clusters").sort_values(ascending=False)
    display(_vc.to_frame())
    display(Markdown(f"행 수: 요약 `{len(summ_df_pv)}` · 트리 `{len(tree_df_pv)}` · 상품 `{len(items_df_pv)}`"))

display(Markdown("### 표: L2 요약 일부"))
summ_show = [c for c in summ_cols if c in summ_df_pv.columns]
tree_show = [c for c in tree_cols if c in tree_df_pv.columns]
item_show = [c for c in item_cols if c in items_df_pv.columns]

if summ_show and not summ_df_pv.empty:
    display(summ_df_pv[summ_show].head(15))
elif not summ_df_pv.empty:
    display(summ_df_pv.head(15))

display(Markdown("### 표: 트리 노드 일부"))

if tree_show and not tree_df_pv.empty:
    display(tree_df_pv.sort_values(["platform", "depth", "node_id"])[tree_show].head(30))
elif not tree_df_pv.empty:
    display(tree_df_pv.head(30))

display(Markdown("### 표: 상품 랜덤 표본"))

if len(items_df_pv):
    sampled = items_df_pv[item_show] if item_show else items_df_pv
    n = min(12, len(sampled))
    part = sampled.sample(n, random_state=0)
    sort_keys = [k for k in ["platform", "cluster_id"] if k in part.columns]
    display(part.sort_values(sort_keys) if sort_keys else part)
else:
    display(items_df_pv)

display(Markdown("### 텍스트 미리보기 (모듈 `stdout_preview_clustering`)"))
print(
    stdout_preview_clustering(summ_df_pv, tree_df_pv, max_summary_rows=12, max_tree_rows=26)
)


클러스터 입력: C:\project\kdtproject\kdtproject\code\backend\src\main\python\TEST\results\_cluster_preview_rows.csv (미리보기: df 상위 800행)
파라미터: {'threshold': 0.35, 'min_shared_tokens': 1, 'threshold_l2': 0.52, 'min_shared_tokens_l2': 2, 'run_second_stage': True, 'token_blacklist_path': WindowsPath('C:/project/kdtproject/kdtproject/code/backend/src/main/python/TEST/blacklist_tokens.csv'), 'print_stdout_preview': False}
입력 파일: C:\project\kdtproject\kdtproject\code\backend\src\main\python\TEST\results\_cluster_preview_rows.csv
상품별 클러스터 결과: C:\project\kdtproject\kdtproject\code\backend\src\main\python\TEST\results\clusters\token_cluster_items_20260517_1616.csv
클러스터 요약 결과: C:\project\kdtproject\kdtproject\code\backend\src\main\python\TEST\results\clusters\token_cluster_summary_20260517_1616.csv
트리 노드 결과: C:\project\kdtproject\kdtproject\code\backend\src\main\python\TEST\results\clusters\token_cluster_tree_20260517_1616.csv

플랫폼별 최종 클러스터(L2 리프) 수
platform
번개장터    320
중고나라    221
Name: cluster_id, dty

### 검증 요약

,leaf_l2_clusters
platform,
번개장터,320
중고나라,221


행 수: 요약 `541` · 트리 `790` · 상품 `800`

### 표: L2 요약 일부

...

,parent_node_id,node_id,platform,depth,item_count,representative_name,cluster_tokens
0,NaN,번개장터::__root,번개장터,0,480,NaN,NaN
1,번개장터::__root,번개장터_001,번개장터,1,1,최저가]ASUS 14인치 게이밍노트북 A14(8845HS/4060/램32,최저가]asus 14인치 게이밍노트북 a14(8845hs/4060/램32
2,번개장터::__root,번개장터_002,번개장터,1,269,HP 오멘 MAX 16 울트라9 RTX5080 램 64기가 게이밍 노트북,게이밍 노트북 hp asus msi i7 레노버 rog 삼성 omen 아수스 16 ...
3,번개장터::__root,번개장터_003,번개장터,1,6,[매입]미개봉 LG그램 갤럭시북 맥북 기업용 대량매입 게이밍 노트북,매입]미개봉 lg그램 갤럭시북 맥북 기업용 대량매입 게이밍 노트북
4,번개장터::__root,번개장터_004,번개장터,1,11,[최저가]HP 16인치 게이밍노트북 오멘(라이젠9 365/5070/램32,게이밍노트북 최저가]hp 16인치 오멘(라이젠9 17인치 365/5070/램32 오...
5,번개장터::__root,번개장터_005,번개장터,1,4,MSI GF63 게이밍노트북 RTX2050/인텔i5/램32GB/윈11프로,msi 게이밍노트북 부품용으로 가져가세요 gf63 rtx2050/인텔i5/램32gb...
6,번개장터::__root,번개장터_006,번개장터,1,1,"msi사이보그 게이밍 노트북 rtx5060, ddr5 16b, 144hz",msi사이보그 게이밍 노트북 rtx5060 ddr5 16b 144hz
7,번개장터::__root,번개장터_007,번개장터,1,2,[무상남음 S급 ] MSI 게이밍노트북 i7 32G RTX4060 1TB,msi 게이밍노트북 i7 rtx4060 풀박 ddr5 144hz 무상남음 s급 32...
8,번개장터::__root,번개장터_008,번개장터,1,3,최저가]기가바이트 16인치 게이밍노트북 어로스(i7-14/4070/1TB,최저가]기가바이트 16인치 게이밍노트북 어로스(i7-14/4070/1tb 에어로(3...
9,번개장터::__root,번개장터_009,번개장터,1,1,[매입]갤럭시북프로 360 맥북 PC 게이밍노트북 24시간 출장방문매입,매입]갤럭시북프로 360 맥북 pc 게이밍노트북 24시간 출장방문매입


### 표: 상품 랜덤 표본

,platform,pid,name,cluster_l1_id,cluster_id,breadcrumb_names,derived_product_name
40,번개장터,407356200,Msi 게이밍 노트북 17인치 팝니다 .롤.서든.피파.옵치.포토샵.캐드,번개장터_002,번개장터_002_s01,HP 오멘 MAX 16 울트라9 RTX5080 램 64기가 게이밍 노트북 › HP ...,HP OMEN 16 게이밍 노트북 RTX 5070 화이트!! 가격다운급매
14,번개장터,406327594,MSI 17인치 게이밍 노트북 4060,번개장터_002,번개장터_002_s01,HP 오멘 MAX 16 울트라9 RTX5080 램 64기가 게이밍 노트북 › HP ...,HP OMEN 16 게이밍 노트북 RTX 5070 화이트!! 가격다운급매
231,번개장터,237701928,(일산) msi 게이밍 노트북 15인치 지포스,번개장터_002,번개장터_002_s120,HP 오멘 MAX 16 울트라9 RTX5080 램 64기가 게이밍 노트북 › (일산...,(일산) msi 게이밍 노트북 15인치 지포스
236,번개장터,394539076,ASUS 게이밍 노트북 / RTX5060 / RAM 32GB / S급,번개장터_002,번개장터_002_s125,HP 오멘 MAX 16 울트라9 RTX5080 램 64기가 게이밍 노트북 › ASU...,ASUS 게이밍 노트북 / RTX5060 / RAM 32GB / S급
240,번개장터,387745500,게이밍 노트북ASUS ROG Strix G15 G512LV,번개장터_002,번개장터_002_s129,HP 오멘 MAX 16 울트라9 RTX5080 램 64기가 게이밍 노트북 › 게이밍...,게이밍 노트북ASUS ROG Strix G15 G512LV
299,번개장터,404407303,S급 MSI 레이더 게이밍 노트북 울트라9 64램 RTX5080 4T,번개장터_010,번개장터_010_s01,S급 MSI 레이더 게이밍 노트북 울트라9 64램 RTX5080 4T,S급 MSI 레이더 게이밍 노트북 울트라9 64램 RTX5080 4T
303,번개장터,407636713,[최저가]MSI 17인치 게이밍노트북 알파(7945HX/4070/1TB),번개장터_013,번개장터_013_s01,[최저가]MSI 17인치 게이밍노트북 알파(7945HX/4070/1TB),[최저가]MSI 17인치 게이밍노트북 알파(7945HX/4070/1TB)
389,번개장터,405620294,MSI 게이밍 노트북 A10 R9 m290x 신품배터리 GX60 3CC,번개장터_067,번개장터_067_s01,MSI 게이밍 노트북 A10 R9 m290x 신품배터리 GX60 3CC,MSI 게이밍 노트북 A10 R9 m290x 신품배터리 GX60 3CC
436,번개장터,406500169,369번 아수스 14인치 그래픽작업용 게이밍 중고노트북,번개장터_104,번개장터_104_s01,369번 아수스 14인치 그래픽작업용 게이밍 중고노트북,369번 아수스 14인치 그래픽작업용 게이밍 중고노트북
500,중고나라,227832236,게이밍 노트북,중고나라_004,중고나라_004_s05,HP 오멘 맥스 16 RTX 5070 ti 신형 게이밍 노트북 작업용 초고성능 OM...,MSI Sword GF76 B13VFK 게이밍 노트북 팝니다


### 텍스트 미리보기 (모듈 `stdout_preview_clustering`)

[L2 리프 요약 · 상위 12 행]
platform    cluster_id  cluster_size                                                       breadcrumb_preview                     derived_product_name                                                           sample_preview
    번개장터  번개장터_002_s01            82 HP 오멘 MAX 16 울트라9 RTX5080 램 64기가 게이밍 노트북 › HP OMEN 16 게이밍 노트북 RTX 5070 화 HP OMEN 16 게이밍 노트북 RTX 5070 화이트!! 가격다운급매 레노버 LEGION 게이밍 노트북 | 에이수스TUF 게이밍 노트북 | i7 게이밍 1660ti 노트북 | ASUS ROG 게이밍 
    번개장터  번개장터_037_s01            11                                               14.5인치 3D 노트북/ssd 탑재/외장그래픽               14.5인치 3D 노트북/ssd 탑재/외장그래픽                                               14.5인치 3D 노트북/ssd 탑재/외장그래픽
    번개장터  번개장터_004_s02             7 [최저가]HP 16인치 게이밍노트북 오멘(라이젠9 365/5070/램32 › 최저가]HP 16인치 게이밍노트북 오멘슬림(i7-14 최저가]HP 16인치 게이밍노트북 오멘슬림(i7-14th/4060/1TB [최저가]HP 16인치 게이밍노트북 오멘(i5-13/4060/QHD) | 최저가]HP 16인치 게이밍노트북 오멘슬림(i7-14th
    번개장터  번개장터_003_s01             6                                    [매입]미개봉 LG그

### 저장된 클러스터 결과만 다시 불러오기

`run_clustering`을 다시 돌리지 않고 **`results/clusters/latest_token_cluster_*.csv`** 만 읽습니다.  
성공 시 위와 동일하게 `summ_df_pv`, `tree_df_pv`, `items_df_pv` 변수가 채워져 이후 탐색에 쓸 수 있습니다.


In [ ]:
from IPython.display import Markdown, display

CLUSTER_RELOAD_BASE = csv_path.resolve().parent / "clusters"
_reload_paths = {
    "summary": CLUSTER_RELOAD_BASE / "latest_token_cluster_summary.csv",
    "tree": CLUSTER_RELOAD_BASE / "latest_token_cluster_tree.csv",
    "items": CLUSTER_RELOAD_BASE / "latest_token_cluster_items.csv",
}

missing = []
for label, path in _reload_paths.items():
    ok = path.is_file()
    print(f"[{'OK' if ok else '없음'}] {label:8s}", path.resolve())
    if not ok:
        missing.append(label)

if not missing:
    summ_df_pv = pd.read_csv(_reload_paths["summary"], encoding="utf-8-sig")
    tree_df_pv = pd.read_csv(_reload_paths["tree"], encoding="utf-8-sig")
    items_df_pv = pd.read_csv(_reload_paths["items"], encoding="utf-8-sig")
    display(
        Markdown(
            "재로드 완료: **`summ_df_pv`**, **`tree_df_pv`**, **`items_df_pv`** (클러스터 테스트 셀과 동일 이름)"
        )
    )
    display(summ_df_pv.head(8))
else:
    display(
        Markdown(
            "위 **클러스터 테스트** 셀을 먼저 실행하거나, `results/clusters/`에 `latest_*.csv`가 있는지 확인하세요."
        )
    )


In [4]:
base_cols = ["keyword", "platform", "pid", "name", "price", "matched_keywords"]

missing = [c for c in base_cols if c not in df.columns]
if missing:
    raise ValueError(f"필요한 컬럼이 없습니다: {missing}")


def tokenize_by_space(text):
    """name을 공백(연속 공백 포함) 기준으로 분리합니다."""
    if pd.isna(text):
        return []
    return [tok for tok in str(text).split() if tok]


token_df = df[base_cols].copy()
token_df["name_tokens"] = token_df["name"].apply(tokenize_by_space)
token_df["token_count"] = token_df["name_tokens"].str.len()
token_df.head(10)

,keyword,platform,pid,name,price,matched_keywords,name_tokens,token_count
0,게이밍 노트북,번개장터,407671617,최저가]ASUS 14인치 게이밍노트북 A14(8845HS/4060/램32,1270000,게이밍 노트북,"[최저가]ASUS, 14인치, 게이밍노트북, A14(8845HS/4060/램32]",4
1,게이밍 노트북,번개장터,396198529,레노버 LEGION 게이밍 노트북,329000,게이밍 노트북,"[레노버, LEGION, 게이밍, 노트북]",4
2,게이밍 노트북,번개장터,399045782,"MSI i7 고성능 게이밍 노트북(램12GB,1.25TB,신품고가배터리)",529000,게이밍 노트북,"[MSI, i7, 고성능, 게이밍, 노트북(램12GB,1.25TB,신품고가배터리)]",5
3,게이밍 노트북,번개장터,407667568,[매입]미개봉 LG그램 갤럭시북 맥북 기업용 대량매입 게이밍 노트북,3000000,"갤럭시, 게이밍 노트북","[[매입]미개봉, LG그램, 갤럭시북, 맥북, 기업용, 대량매입, 게이밍, 노트북]",8
4,게이밍 노트북,번개장터,407666216,[최저가]HP 16인치 게이밍노트북 오멘(라이젠9 365/5070/램32,2150000,게이밍 노트북,"[[최저가]HP, 16인치, 게이밍노트북, 오멘(라이젠9, 365/5070/램32]",5
5,게이밍 노트북,번개장터,406964298,에이수스TUF 게이밍 노트북,640000,게이밍 노트북,"[에이수스TUF, 게이밍, 노트북]",3
6,게이밍 노트북,번개장터,407664491,MSI 게이밍 노트북 펄스 17 rtx4070,1200000,게이밍 노트북,"[MSI, 게이밍, 노트북, 펄스, 17, rtx4070]",6
7,게이밍 노트북,번개장터,398590249,msi 게이밍노트북 부품용으로 가져가세요^^!!,200000,게이밍 노트북,"[msi, 게이밍노트북, 부품용으로, 가져가세요^^!!]",4
8,게이밍 노트북,번개장터,407661175,[최저가]HP 16인치 게이밍노트북 오멘(i5-13/4060/QHD),1090000,게이밍 노트북,"[[최저가]HP, 16인치, 게이밍노트북, 오멘(i5-13/4060/QHD)]",4
9,게이밍 노트북,번개장터,407611942,HP Victus 게이밍 노트북 i5 RTX3050ti,750000,게이밍 노트북,"[HP, Victus, 게이밍, 노트북, i5, RTX3050ti]",6


In [5]:
TOP_N = 50

long_tokens = token_df.explode("name_tokens").rename(columns={"name_tokens": "token"})
long_tokens = long_tokens.dropna(subset=["token"])

occurrence_counts = long_tokens.groupby("token").size().rename("occurrence_count")
product_counts = long_tokens.drop_duplicates(subset=["pid", "token"]).groupby("token").size().rename("product_count")

freq_df = pd.concat([occurrence_counts, product_counts], axis=1).sort_values(
    "occurrence_count", ascending=False
)
freq_df.head(TOP_N)

,occurrence_count,product_count
token,,
갤럭시,4227,4222
중고폰,1811,1797
무잔상,1582,1582
핫토이,1562,1560
세이코,1386,1383
블랙,1285,1285
스텔라이브,1250,1250
삼성,1072,1072
정상공기기,1047,1047


In [6]:
TOP_PER_PLATFORM = 20

occ_by_platform = (
    long_tokens.groupby(["platform", "token"])
    .size()
    .rename("occurrence_count")
    .reset_index()
)

product_by_platform = (
    long_tokens.drop_duplicates(subset=["platform", "pid", "token"])
    .groupby(["platform", "token"])
    .size()
    .rename("product_count")
    .reset_index()
)

freq_by_platform = occ_by_platform.merge(product_by_platform, on=["platform", "token"])
freq_by_platform = freq_by_platform.sort_values(
    ["platform", "occurrence_count"], ascending=[True, False]
)

# 플랫폼별 전체 빈도표: freq_by_platform_for["번개장터"].head(30)
freq_by_platform_for = {
    plat: group.reset_index(drop=True)
    for plat, group in freq_by_platform.groupby("platform", sort=True)
}

# 조회 조건 (기본: 플랫폼별 상위 TOP_PER_PLATFORM 토큰 미리보기)
QUERY_PLATFORM = None  # 예: "번개장터", "중고나라"
TOKEN_CONTAINS = None  # 예: "갤럭시"

if QUERY_PLATFORM is None and TOKEN_CONTAINS is None:
    platform_token_preview = freq_by_platform.groupby("platform", group_keys=False).head(
        TOP_PER_PLATFORM
    )
else:
    q = freq_by_platform
    if QUERY_PLATFORM is not None:
        q = q.loc[q["platform"] == QUERY_PLATFORM]
    if TOKEN_CONTAINS:
        q = q.loc[q["token"].astype(str).str.contains(TOKEN_CONTAINS, regex=False, na=False)]
    platform_token_preview = q.head(TOP_PER_PLATFORM)

if platform_token_preview.empty:
    print("조회 결과가 비어 있습니다.")
    print("- CSV에 해당 플랫폼/토큰이 없거나, 위 셀(token_df/long_tokens)을 먼저 실행했는지 확인하세요.")
    if "df" in dir() and hasattr(df, "columns") and "platform" in df.columns:
        print("- 현재 df의 platform:", df["platform"].dropna().unique().tolist())
    print("- freq_by_platform_for 키:", list(freq_by_platform_for.keys()))

platform_token_preview

,platform,token,occurrence_count,product_count
7538,번개장터,갤럭시,4032,4027
16325,번개장터,중고폰,1771,1757
11053,번개장터,무잔상,1549,1549
18986,번개장터,핫토이,1156,1155
12089,번개장터,블랙,1155,1155
15959,번개장터,정상공기기,1047,1047
13389,번개장터,스텔라이브,1036,1036
12808,번개장터,세이코,1015,1013
12477,번개장터,삼성,984,984
2167,번개장터,256GB,714,714


In [7]:
JOONGNA = "중고나라"

if JOONGNA in freq_by_platform_for:
    freq_by_platform_for[JOONGNA].head(TOP_PER_PLATFORM)
else:
    print(f"'{JOONGNA}' 행이 없습니다. Cell 1에서 통합 CSV를 읽거나, 아래 중 실제 있는 플랫폼을 쓰세요.")
    print(list(freq_by_platform_for.keys()))